# Data Fusion Track 2 — 0.86+ push (Kaggle-adapted)

Основа взята из `d-ulybin/data_fusion_track_2` (public 0.8536), дополнена более агрессивным суперблендингом:
- per-target rank blend с более мелкой сеткой;
- meta-combo: blend + stacking + (optional) lgbm_meta;
- holdout-оптимизация весов по target-ам;
- финальный super submission `submissions/superblend.parquet`.


In [1]:

# 1) Setup: clone solution repo + install deps + patch paths for Kaggle
import os, re, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/d-ulybin/data_fusion_track_2"
REPO_DIR = Path("/kaggle/working/data_fusion_track_2")
DATA_DIR = "/kaggle/input/datasets/hatab123/data-fusion-contest-2026/"  # <-- Kaggle input path

if REPO_DIR.exists():
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=False)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=False)

# patch DATA_DIR and some blend granularity for stronger search
utils_path = REPO_DIR / "utils.py"
s = utils_path.read_text(encoding="utf-8")
s = re.sub(r'DATA_DIR\s*=\s*"[^"]+"', f'DATA_DIR = "{DATA_DIR}"', s)
utils_path.write_text(s, encoding="utf-8")

# finer search in blending/stacking scripts
for rel in ["06_blend.py", "07_stacking.py"]:
    p = REPO_DIR / rel
    t = p.read_text(encoding="utf-8")
    t = t.replace("step=0.10", "step=0.05")
    t = t.replace("np.arange(0, 1.05, 0.05)", "np.arange(0, 1.02, 0.02)")
    p.write_text(t, encoding="utf-8")

print("Repo ready:", REPO_DIR)
print("DATA_DIR set to:", DATA_DIR)


Cloning into '/kaggle/working/data_fusion_track_2'...


Repo ready: /kaggle/working/data_fusion_track_2
DATA_DIR set to: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/


In [ ]:
# 2) Pipeline:
# 01 -> (02 & 04 parallel, NO LIMITS) -> 03 FAST (NO LIMITS) -> 05 FAST (GPU->CPU fallback) -> 06 -> 07
# Минимум вывода: всё пишет в logs_parallel/*.log

import os, re, subprocess, sys, shutil, textwrap, time
from pathlib import Path

REPO_DIR = Path("/kaggle/working/data_fusion_track_2")
os.chdir(REPO_DIR)

LOG_DIR = REPO_DIR / "logs_parallel"
LOG_DIR.mkdir(exist_ok=True)

# ===== настройки =====
FORCE_RETRAIN_03 = False   # True => удалит checkpoints_lgbm/lgbm_predictions.npz и submissions/lgbm.parquet
FORCE_RETRAIN_04 = False   # True => удалит checkpoints_pyboost/pyboost_predictions.npz и submissions/pyboost.parquet
FORCE_RETRAIN_05 = True    # True => удалит checkpoints_lgbm_meta/lgbm_predictions.npz и submissions/lgbm_meta.parquet

# ---------------------------
# helpers
# ---------------------------
def run(cmd, check=False):
    return subprocess.run(["bash", "-lc", cmd], check=check)

def exists(p: Path) -> bool:
    return p.exists()

def safe_unlink(p: Path):
    if p.exists():
        p.unlink()

def safe_backup(p: Path):
    if p.exists():
        bak = p.with_suffix(p.suffix + ".bak")
        if not bak.exists():
            shutil.copy2(p, bak)

def run_logged(script: str, log_path: Path, check=True, env=None):
    log_path.parent.mkdir(exist_ok=True, parents=True)
    with open(log_path, "w", buffering=1) as f:
        p = subprocess.run([sys.executable, script], cwd=str(REPO_DIR), stdout=f, stderr=subprocess.STDOUT, env=env)
    if check and p.returncode != 0:
        # покажем хвост лога при ошибке
        try:
            tail = subprocess.check_output(["bash", "-lc", f"tail -n 120 {log_path}"], text=True)
        except Exception:
            tail = "(tail failed)"
        raise RuntimeError(f"{script} failed rc={p.returncode}\n--- tail {log_path.name} ---\n{tail}")
    return p.returncode

def run_parallel_no_limits(scripts):
    """Параллельный запуск без каких-либо ограничений CPU/GPU. Вывод в логи."""
    procs = {}
    for s in scripts:
        log_path = LOG_DIR / f"{s}.log"
        f = open(log_path, "w", buffering=1)
        p = subprocess.Popen([sys.executable, s], cwd=str(REPO_DIR), stdout=f, stderr=subprocess.STDOUT)
        procs[s] = (p, f, log_path)
        print(f"[PARALLEL] started {s} -> {log_path.name}")

    # ждать завершения, без спама tail
    while True:
        alive = [s for s,(p,_,_) in procs.items() if p.poll() is None]
        if not alive:
            break
        print("[PARALLEL] running:", alive)
        time.sleep(60)

    failed = []
    for s,(p,f,lp) in procs.items():
        rc = p.poll()
        f.close()
        print(f"[PARALLEL] done {s} rc={rc} (log {lp.name})")
        if rc != 0:
            failed.append((s, rc, lp))

    if failed:
        msg = []
        for s, rc, lp in failed:
            try:
                tail = subprocess.check_output(["bash", "-lc", f"tail -n 160 {lp}"], text=True)
            except Exception:
                tail = "(tail failed)"
            msg.append(f"\n{s} rc={rc}\n--- tail {lp.name} ---\n{tail}")
        raise RuntimeError("Some parallel jobs failed:" + "".join(msg))

# ---------------------------
# LightGBM OpenCL GPU detect
# ---------------------------
def detect_opencl_ids():
    run("apt-get update -y")
    run("apt-get install -y --no-install-recommends clinfo ocl-icd-opencl-dev opencl-headers")
    run("apt-get install -y --no-install-recommends nvidia-opencl-icd || true")
    out = subprocess.check_output(["bash", "-lc", "clinfo -l"], text=True, stderr=subprocess.STDOUT)

    # парсим платформу/девайс
    platforms = []
    pid = None
    pname = None
    devs = []
    for line in out.splitlines():
        line = line.strip()
        mplat = re.match(r"Platform\s+#(\d+):\s*(.*)", line)
        if mplat:
            if pid is not None:
                platforms.append((pid, pname, devs))
            pid = int(mplat.group(1))
            pname = mplat.group(2).strip()
            devs = []
            continue
        mdev = re.match(r"`--\s*Device\s+#(\d+):\s*(.*)", line)
        if mdev and pid is not None:
            devs.append((int(mdev.group(1)), mdev.group(2).strip()))
    if pid is not None:
        platforms.append((pid, pname, devs))

    for pid, pname, devs in platforms:
        if "NVIDIA" in pname.upper() and devs:
            return pid, devs[0][0]
    if platforms and platforms[0][2]:
        return platforms[0][0], platforms[0][2][0][0]
    return -1, -1

def smoke_lgbm_gpu_opencl(platform_id: int, device_id: int) -> bool:
    try:
        import lightgbm as lgb
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=5000, n_features=64, random_state=42)
        params = dict(
            objective="binary",
            metric="auc",
            learning_rate=0.1,
            num_leaves=64,
            verbosity=-1,
            device_type="gpu",
            gpu_platform_id=int(platform_id),
            gpu_device_id=int(device_id),
            gpu_use_dp=False,
            max_bin=63,
            min_data_in_bin=1,
        )
        booster = lgb.train(params, lgb.Dataset(X, label=y), num_boost_round=10)
        _ = booster.predict(X[:1000])
        return True
    except Exception:
        return False

OPENCL_PLATFORM, OPENCL_DEVICE = detect_opencl_ids()
DEVICE_TYPE_03 = "gpu" if smoke_lgbm_gpu_opencl(OPENCL_PLATFORM, OPENCL_DEVICE) else "cpu"
print(f"[INFO] LGBM step03 device_type={DEVICE_TYPE_03} (OpenCL {OPENCL_PLATFORM}/{OPENCL_DEVICE})")

# ---------------------------
# Ensure PyBoost deps (treelite<4!)
# ---------------------------
def ensure_pyboost():
    run(f"{sys.executable} -m pip install -q -U cupy-cuda12x py-boost", check=True)
    run(f"{sys.executable} -m pip install -q -U 'treelite>=3,<4'", check=False)

ensure_pyboost()
print("[INFO] PyBoost deps OK (treelite<4 enforced)")

# ---------------------------
# Patch Step03 (FAST)
# ---------------------------
safe_backup(REPO_DIR / "03_train_lgbm.py")

fast_03_template = r'''
"""Step 3: Train LightGBM (FAST multi-target, 5-fold)."""

import gc, json, os, sys, time, warnings
from pathlib import Path
import lightgbm as lgb
import numpy as np
import polars as pl
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from utils import SEED, DATA_DIR, N_FOLDS, compute_macro_auc, verify_submission

FEATURES_DIR = Path("features")
CHECKPOINT_DIR = Path("checkpoints_lgbm")

BASE_PARAMS = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.050216,
    num_leaves=34,
    max_depth=10,
    min_child_samples=102,
    subsample=0.521715,
    colsample_bytree=0.205092,
    reg_alpha=8.146025,
    reg_lambda=7.761833,
    min_split_gain=0.424512,
    subsample_freq=2,
    seed=SEED,
    verbosity=-1,
    force_col_wise=True,
)

NUM_BOOST_ROUND = 1500
EARLY_STOPPING_ROUNDS = 100

DEVICE_TYPE = "__DEVICE_TYPE__"
OPENCL_PLATFORM = __OPENCL_PLATFORM__
OPENCL_DEVICE = __OPENCL_DEVICE__

def main():
    t0 = time.time()
    print("=" * 60)
    print("Step 3: Train LightGBM (FAST, 5-fold × 41 targets)")
    print("=" * 60)

    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    feature_cols = meta["feature_names"]
    cat_feature_names = meta["cat_cols"]
    target_cols = meta["target_cols"]
    cat_indices = [feature_cols.index(c) for c in cat_feature_names]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

    X_train = np.ascontiguousarray(train_feat.drop("customer_id").to_numpy(), dtype=np.float32)
    X_test = np.ascontiguousarray(test_feat.drop("customer_id").to_numpy(), dtype=np.float32)
    y_train = train_tgt.select(target_cols).to_numpy().astype(np.float32)

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CHECKPOINT_DIR / "lgbm_predictions.npz"
    if cache_file.exists():
        print(f"Predictions exist at {cache_file}. Skip.")
        return

    params = BASE_PARAMS.copy()
    params["device_type"] = DEVICE_TYPE
    if DEVICE_TYPE == "gpu":
        params.update(
            gpu_platform_id=int(OPENCL_PLATFORM),
            gpu_device_id=int(OPENCL_DEVICE),
            gpu_use_dp=False,
            max_bin=63,
            min_data_in_bin=1,
        )
    else:
        params.update(max_bin=255)

    n_train, n_test = X_train.shape[0], X_test.shape[0]
    n_targets = len(target_cols)
    oof_preds = np.zeros((n_train, n_targets), dtype=np.float32)
    test_sum = np.zeros((n_test, n_targets), dtype=np.float32)
    fold_aucs = []

    kf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(np.arange(n_train), y_train)):
        X_tr = X_train[tr_idx]
        X_val = X_train[val_idx]
        y_tr = y_train[tr_idx]
        y_val = y_train[val_idx]

        train_data = lgb.Dataset(X_tr, label=y_tr[:, 0], categorical_feature=cat_indices, free_raw_data=False)
        valid_data = lgb.Dataset(X_val, label=y_val[:, 0], categorical_feature=cat_indices, reference=train_data, free_raw_data=False)

        fold_test = np.zeros((n_test, n_targets), dtype=np.float32)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            for i in range(n_targets):
                train_data.set_label(y_tr[:, i])
                valid_data.set_label(y_val[:, i])

                booster = lgb.train(
                    params=params,
                    train_set=train_data,
                    num_boost_round=NUM_BOOST_ROUND,
                    valid_sets=[valid_data],
                    valid_names=["valid"],
                    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(period=0)],
                )
                best_iter = booster.best_iteration or NUM_BOOST_ROUND
                oof_preds[val_idx, i] = booster.predict(X_val, num_iteration=best_iter).astype(np.float32)
                fold_test[:, i] = booster.predict(X_test, num_iteration=best_iter).astype(np.float32)
                del booster

        fold_auc, _ = compute_macro_auc(y_val, oof_preds[val_idx], target_cols)
        fold_aucs.append(float(fold_auc))
        test_sum += fold_test

        del X_tr, X_val, y_tr, y_val, train_data, valid_data, fold_test
        gc.collect()

        print(f"Fold {fold_idx+1}/{N_FOLDS} done, AUC={fold_auc:.4f}")

    test_avg = test_sum / N_FOLDS
    oof_auc, _ = compute_macro_auc(y_train, oof_preds, target_cols)
    np.savez(cache_file, oof_preds=oof_preds, test_preds=test_avg, fold_aucs=np.array(fold_aucs))
    print(f"Saved: {cache_file}, OOF={oof_auc:.4f}")

    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_avg.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    Path("submissions").mkdir(exist_ok=True)
    submit.write_parquet("submissions/lgbm.parquet")
    print("Saved: submissions/lgbm.parquet")

if __name__ == "__main__":
    os.environ["PYTHONUNBUFFERED"] = "1"
    np.random.seed(SEED)
    main()
'''
fast_03 = (fast_03_template
           .replace("__DEVICE_TYPE__", DEVICE_TYPE_03)
           .replace("__OPENCL_PLATFORM__", str(int(OPENCL_PLATFORM)))
           .replace("__OPENCL_DEVICE__", str(int(OPENCL_DEVICE))))
(REPO_DIR / "03_train_lgbm.py").write_text(textwrap.dedent(fast_03), encoding="utf-8")

if FORCE_RETRAIN_03:
    safe_unlink(REPO_DIR / "checkpoints_lgbm" / "lgbm_predictions.npz")
    safe_unlink(REPO_DIR / "submissions" / "lgbm.parquet")

print("[PATCH] Step03 FAST written")

# ---------------------------
# Patch Step05 (FAST + GPU->CPU fallback on 'bin size ... cannot run on GPU')
# ---------------------------
safe_backup(REPO_DIR / "05_train_lgbm_meta.py")

fast_05_template = r'''
"""Step 5: LGBM meta (FAST).
- NN OOF: vectorized
- Binning reduced: per-target Dataset + subset folds
- Tries GPU(OpenCL) first, but falls back to CPU if GPU fails with 'bin size ... cannot run on GPU'
"""

import gc, json, os, sys, time, warnings
from pathlib import Path

import lightgbm as lgb
from lightgbm.basic import LightGBMError
import numpy as np
import polars as pl
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from utils import SEED, DATA_DIR, N_FOLDS, compute_macro_auc, verify_submission

FEATURES_DIR = Path("features")
CHECKPOINT_DIR = Path("checkpoints_lgbm_meta")

BASE_PARAMS = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.050216,
    num_leaves=34,
    max_depth=10,
    min_child_samples=102,
    subsample=0.521715,
    colsample_bytree=0.205092,
    reg_alpha=8.146025,
    reg_lambda=7.761833,
    min_split_gain=0.424512,
    subsample_freq=2,
    seed=SEED,
    verbosity=-1,
    force_col_wise=True,
)

NUM_BOOST_ROUND = 1500
EARLY_STOPPING_ROUNDS = 100

PREFERRED_DEVICE = "__PREFERRED_DEVICE__"     # 'gpu' or 'cpu'
OPENCL_PLATFORM = __OPENCL_PLATFORM__
OPENCL_DEVICE = __OPENCL_DEVICE__

def load_model_predictions(n_train: int, n_targets: int):
    nn_dir = Path("checkpoints_nn")
    nn_oof = np.zeros((n_train, n_targets), dtype=np.float32)
    test_parts = []
    for fi in range(N_FOLDS):
        d = np.load(nn_dir / f"fold_{fi}.npz")
        val_idx = d["val_idx"].astype(np.int64)
        nn_oof[val_idx] = d["val_preds"].astype(np.float32)
        test_parts.append(d["test_preds"].astype(np.float32))
    nn_test = np.mean(np.stack(test_parts, axis=0), axis=0).astype(np.float32)

    d = np.load("checkpoints_lgbm/lgbm_predictions.npz")
    lgbm_oof = d["oof_preds"].astype(np.float32)
    lgbm_test = d["test_preds"].astype(np.float32)

    d = np.load("checkpoints_pyboost/pyboost_predictions.npz")
    pb_oof = d["oof_preds"].astype(np.float32)
    pb_test = d["test_preds"].astype(np.float32)

    meta_train = np.concatenate([lgbm_oof, nn_oof, pb_oof], axis=1)
    meta_test = np.concatenate([lgbm_test, nn_test, pb_test], axis=1)
    return meta_train, meta_test

def build_params(device_type: str):
    params = BASE_PARAMS.copy()
    params["device_type"] = device_type
    if device_type == "gpu":
        # max_bin <= 255, но категориальные бинны могут быть >255 -> тогда упадёт, мы поймаем и переключимся на CPU
        params.update(
            gpu_platform_id=int(OPENCL_PLATFORM),
            gpu_device_id=int(OPENCL_DEVICE),
            gpu_use_dp=False,
            max_bin=63,
            min_data_in_bin=1,
        )
    else:
        params.update(max_bin=255)
    return params

def train_all(params, X_train_full, X_test_full, y_train, cat_indices, target_cols):
    n_train = X_train_full.shape[0]
    n_test = X_test_full.shape[0]
    n_targets = len(target_cols)

    n_base = X_train_full.shape[1] - 3 * n_targets
    own_cols = np.array([[n_base + i, n_base + n_targets + i, n_base + 2*n_targets + i] for i in range(n_targets)], dtype=np.int64)

    kf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    all_idx = np.arange(n_train)
    splits = list(kf.split(all_idx, y_train))

    oof = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    for ti in range(n_targets):
        cols = own_cols[ti]

        saved_tr = X_train_full[:, cols].copy()
        saved_te = X_test_full[:, cols].copy()
        X_train_full[:, cols] = np.nan
        X_test_full[:, cols] = np.nan

        full_ds = lgb.Dataset(X_train_full, label=y_train[:, ti], categorical_feature=cat_indices, free_raw_data=False)

        t_sum = np.zeros((n_test,), dtype=np.float32)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            for (tr_idx, val_idx) in splits:
                train_ds = full_ds.subset(tr_idx)
                valid_ds = full_ds.subset(val_idx)

                booster = lgb.train(
                    params=params,
                    train_set=train_ds,
                    num_boost_round=NUM_BOOST_ROUND,
                    valid_sets=[valid_ds],
                    valid_names=["valid"],
                    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(period=0)],
                )
                best_iter = booster.best_iteration or NUM_BOOST_ROUND

                oof[val_idx, ti] = booster.predict(X_train_full[val_idx], num_iteration=best_iter).astype(np.float32)
                t_sum += booster.predict(X_test_full, num_iteration=best_iter).astype(np.float32)

                del booster, train_ds, valid_ds
                gc.collect()

        test_pred[:, ti] = t_sum / N_FOLDS

        X_train_full[:, cols] = saved_tr
        X_test_full[:, cols] = saved_te
        del saved_tr, saved_te, full_ds, t_sum
        gc.collect()

        if (ti + 1) % 5 == 0 or ti == n_targets - 1:
            print(f"target {ti+1}/{n_targets} done")

    return oof, test_pred

def main():
    print("=" * 60)
    print("Step 5: LGBM meta (FAST) with GPU->CPU fallback")
    print("=" * 60)

    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    feature_cols = meta["feature_names"]
    cat_feature_names = meta["cat_cols"]
    target_cols = meta["target_cols"]
    cat_indices = [feature_cols.index(c) for c in cat_feature_names]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

    X_train_base = np.ascontiguousarray(train_feat.drop("customer_id").to_numpy(), dtype=np.float32)
    X_test_base = np.ascontiguousarray(test_feat.drop("customer_id").to_numpy(), dtype=np.float32)
    y_train = train_tgt.select(target_cols).to_numpy().astype(np.float32)

    n_train = X_train_base.shape[0]
    n_targets = len(target_cols)

    meta_train, meta_test = load_model_predictions(n_train=n_train, n_targets=n_targets)
    X_train_full = np.ascontiguousarray(np.concatenate([X_train_base, meta_train], axis=1), dtype=np.float32)
    X_test_full = np.ascontiguousarray(np.concatenate([X_test_base, meta_test], axis=1), dtype=np.float32)
    del X_train_base, X_test_base, meta_train, meta_test
    gc.collect()

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CHECKPOINT_DIR / "lgbm_predictions.npz"
    if cache_file.exists():
        print(f"Predictions exist at {cache_file}. Skip.")
        return

    # try preferred device first
    params = build_params(PREFERRED_DEVICE)
    try:
        oof, test_pred = train_all(params, X_train_full, X_test_full, y_train, cat_indices, target_cols)
    except LightGBMError as e:
        msg = str(e).lower()
        if PREFERRED_DEVICE == "gpu" and ("bin size" in msg and "gpu" in msg):
            print("[WARN] GPU failed due to bin size. Fallback to CPU.")
            params = build_params("cpu")
            oof, test_pred = train_all(params, X_train_full, X_test_full, y_train, cat_indices, target_cols)
        else:
            raise

    oof_auc, _ = compute_macro_auc(y_train, oof, target_cols)
    np.savez(cache_file, oof_preds=oof, test_preds=test_pred)
    print(f"Saved: {cache_file}, OOF={oof_auc:.4f}")

    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_pred.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    Path("submissions").mkdir(exist_ok=True)
    submit.write_parquet("submissions/lgbm_meta.parquet")
    print("Saved: submissions/lgbm_meta.parquet")

if __name__ == "__main__":
    os.environ["PYTHONUNBUFFERED"] = "1"
    np.random.seed(SEED)
    main()
'''
# Step05: пытаемся GPU сначала (но с fallback на CPU)
PREFERRED_DEVICE_05 = "gpu"

fast_05 = (fast_05_template
           .replace("__PREFERRED_DEVICE__", PREFERRED_DEVICE_05)
           .replace("__OPENCL_PLATFORM__", str(int(OPENCL_PLATFORM)))
           .replace("__OPENCL_DEVICE__", str(int(OPENCL_DEVICE))))
(REPO_DIR / "05_train_lgbm_meta.py").write_text(textwrap.dedent(fast_05), encoding="utf-8")

if FORCE_RETRAIN_05:
    safe_unlink(REPO_DIR / "checkpoints_lgbm_meta" / "lgbm_predictions.npz")
    safe_unlink(REPO_DIR / "submissions" / "lgbm_meta.parquet")

print("[PATCH] Step05 FAST written (GPU->CPU fallback enabled)")

# ---------------------------
# Run graph:
# 01 -> (02 & 04 parallel NO LIMITS) -> 03 NO LIMITS -> 05 (fallback) -> 06 -> 07
# ---------------------------
HAVE_01 = exists(REPO_DIR / "features" / "train_features.parquet") and exists(REPO_DIR / "features" / "meta.json")
HAVE_NN_FOLDS = exists(REPO_DIR / "checkpoints_nn" / "fold_0.npz")
HAVE_02 = exists(REPO_DIR / "submissions" / "nn.parquet") and HAVE_NN_FOLDS
HAVE_03 = exists(REPO_DIR / "checkpoints_lgbm" / "lgbm_predictions.npz")
HAVE_04 = exists(REPO_DIR / "checkpoints_pyboost" / "pyboost_predictions.npz")
HAVE_05 = exists(REPO_DIR / "checkpoints_lgbm_meta" / "lgbm_predictions.npz")
HAVE_06 = exists(REPO_DIR / "submissions" / "blend.parquet")
HAVE_07 = exists(REPO_DIR / "submissions" / "stacking.parquet")

print("\n[STATE] features:", HAVE_01, "| nn:", HAVE_02, "| lgbm:", HAVE_03, "| pyboost:", HAVE_04, "| meta:", HAVE_05)

# 01
if not HAVE_01:
    print("[RUN] 01_feature_engineering.py")
    run_logged("01_feature_engineering.py", LOG_DIR / "01_feature_engineering.py.log", check=True)
else:
    print("[SKIP] 01")

# 02 & 04 parallel (NO LIMITS)
to_par = []
if not HAVE_02:
    to_par.append("02_train_nn.py")
if not HAVE_04:
    if FORCE_RETRAIN_04:
        safe_unlink(REPO_DIR / "checkpoints_pyboost" / "pyboost_predictions.npz")
        safe_unlink(REPO_DIR / "submissions" / "pyboost.parquet")
    to_par.append("04_train_pyboost.py")

if to_par:
    print("[RUN] parallel:", to_par)
    run_parallel_no_limits(to_par)
else:
    print("[SKIP] 02 & 04")

# 03 (NO LIMITS, после 02/04)
HAVE_03 = exists(REPO_DIR / "checkpoints_lgbm" / "lgbm_predictions.npz")
if not HAVE_03:
    print("[RUN] 03_train_lgbm.py (no limits)")
    run_logged("03_train_lgbm.py", LOG_DIR / "03_train_lgbm.py.log", check=True)
else:
    print("[SKIP] 03")

# 05 (FAST, GPU->CPU fallback)
HAVE_NN_FOLDS = exists(REPO_DIR / "checkpoints_nn" / "fold_0.npz")
HAVE_02 = exists(REPO_DIR / "submissions" / "nn.parquet") and HAVE_NN_FOLDS
HAVE_03 = exists(REPO_DIR / "checkpoints_lgbm" / "lgbm_predictions.npz")
HAVE_04 = exists(REPO_DIR / "checkpoints_pyboost" / "pyboost_predictions.npz")
if not (HAVE_02 and HAVE_03 and HAVE_04):
    raise RuntimeError("Missing deps for step05: need 02 (with checkpoints_nn/fold_*.npz), 03, 04")

HAVE_05 = exists(REPO_DIR / "checkpoints_lgbm_meta" / "lgbm_predictions.npz")
if not HAVE_05:
    print("[RUN] 05_train_lgbm_meta.py (gpu->cpu fallback)")
    run_logged("05_train_lgbm_meta.py", LOG_DIR / "05_train_lgbm_meta.py.log", check=True)
else:
    print("[SKIP] 05")

# 06 / 07
if not exists(REPO_DIR / "submissions" / "blend.parquet"):
    print("[RUN] 06_blend.py")
    run_logged("06_blend.py", LOG_DIR / "06_blend.py.log", check=True)
else:
    print("[SKIP] 06")

if not exists(REPO_DIR / "submissions" / "stacking.parquet"):
    print("[RUN] 07_stacking.py")
    run_logged("07_stacking.py", LOG_DIR / "07_stacking.py.log", check=True)
else:
    print("[SKIP] 07")

print("\nDONE. Logs in:", LOG_DIR)
print("If something failed, open the corresponding *.log in logs_parallel/")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,810 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/m

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
clinfo is already the newest version (3.0.21.02.21-1).
ocl-icd-opencl-dev is already the newest version (2.2.14-3).
ocl-icd-opencl-dev set to manually installed.
The following NEW packages will be installed:
  opencl-headers
0 upgraded, 1 newly installed, 0 to remove and 173 not upgraded.
Need to get 1,754 B of archives.
After this operation, 12.3 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 opencl-headers all 3.0~2022.01.04-1 [1,754 B]
Fetched 1,754 B in 0s (6,737 B/s)
Selecting previously unselected package opencl-headers.
(Reading database ... 124463 files and directories currently installed.)
Preparing to unpack .../opencl-headers_3.0~2022.01.04-1_all.deb ...
Unpacking opencl-headers (3.0~2022.01.04-1) ...
Setting up opencl-headers (3.0~2022.01.04-1) ...
Reading package lists...
Building dependency tree...
Reading state information...
Package nvidia-opencl-icd is a vir

E: Package 'nvidia-opencl-icd' has no installation candidate
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[INFO] LGBM step03 device_type=gpu (OpenCL 0/0)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 14.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 25.10.0 requires treelite==4.4.1, but you have treelite 3.9.1 which is incompatible.


[INFO] PyBoost deps OK (treelite<4 enforced)
[PATCH] Step03 FAST written
[PATCH] Step05 FAST written (GPU->CPU fallback enabled)

[STATE] features: False | nn: False | lgbm: False | pyboost: False | meta: False
[RUN] 01_feature_engineering.py
[RUN] parallel: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] started 02_train_nn.py -> 02_train_nn.py.log
[PARALLEL] started 04_train_pyboost.py -> 04_train_pyboost.py.log
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running: ['02_train_nn.py', '04_train_pyboost.py']
[PARALLEL] running:

KeyboardInterrupt: 

In [ ]:

# 3) Superblend (new): blend + stacking + meta with per-target optimized weights on OOF
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

DATA_DIR = "/kaggle/input/datasets/hatab123/data-fusion-contest-2026/"
ROOT = Path("/kaggle/working/data_fusion_track_2")

train_tgt = pl.read_parquet(f"{DATA_DIR}train_target.parquet")
target_cols = [c for c in train_tgt.columns if c.startswith("target_")]
y = train_tgt.select(target_cols).to_numpy().astype(np.float32)
n_targets = len(target_cols)

# Load OOF/test preds from artifacts
# Required upstream artifacts:
# - blend_artifacts/blend_data.npz (nn/lgbm/pyboost oof/test)
# - submissions/blend.parquet
# - submissions/stacking.parquet
d = np.load(ROOT / "blend_artifacts" / "blend_data.npz")
oof_nn, test_nn = d["oof_nn"], d["test_nn"]
oof_lgbm, test_lgbm = d["oof_lgbm"], d["test_lgbm"]
oof_pb, test_pb = d["oof_pb"], d["test_pb"]

sub_blend = pl.read_parquet(ROOT / "submissions" / "blend.parquet")
sub_stack = pl.read_parquet(ROOT / "submissions" / "stacking.parquet")
predict_cols = [c.replace("target_", "predict_") for c in target_cols]
test_blend = sub_blend.select(predict_cols).to_numpy()
test_stack = sub_stack.select(predict_cols).to_numpy()

# Rebuild baseline oof blend with finer grid

def to_ranks(a):
    return np.column_stack([rankdata(a[:, i]) for i in range(a.shape[1])]).astype(np.float64)

def safe_auc(y_true, y_pred):
    if np.unique(y_true).size < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_pred))

r_nn, r_lgbm, r_pb = to_ranks(oof_nn), to_ranks(oof_lgbm), to_ranks(oof_pb)
rt_nn, rt_lgbm, rt_pb = to_ranks(test_nn), to_ranks(test_lgbm), to_ranks(test_pb)

oof_base = np.zeros_like(r_nn)
test_base = np.zeros_like(rt_nn)

grid = np.arange(0.0, 1.0001, 0.05)
for j in range(n_targets):
    yj = y[:, j]
    best_auc = -1
    best_w = (1/3, 1/3, 1/3)
    for w0 in grid:
        for w1 in grid:
            if w0 + w1 > 1.0:
                continue
            w2 = 1.0 - w0 - w1
            pred = w0 * r_nn[:, j] + w1 * r_lgbm[:, j] + w2 * r_pb[:, j]
            auc = safe_auc(yj, pred)
            if auc > best_auc:
                best_auc = auc
                best_w = (w0, w1, w2)
    w0, w1, w2 = best_w
    oof_base[:, j] = w0 * r_nn[:, j] + w1 * r_lgbm[:, j] + w2 * r_pb[:, j]
    test_base[:, j] = w0 * rt_nn[:, j] + w1 * rt_lgbm[:, j] + w2 * rt_pb[:, j]

# rank versions of stack/blend submissions
# (for oof proxies use base + simple synthetic: here we use base as robust anchor)
# If you have oof stack saved separately, replace this proxy with true oof_stack.
r_test_blend = to_ranks(test_blend)
r_test_stack = to_ranks(test_stack)

# Super-combo for test: heavier on stacking, keep base anchor
# NOTE: weights chosen from prior experiments; adjust if you keep true oof_stack.
w_base, w_stack, w_blend = 0.30, 0.55, 0.15
test_super = w_base * test_base + w_stack * r_test_stack + w_blend * r_test_blend

sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
sub = pl.DataFrame({"customer_id": sample["customer_id"]}).hstack(
    pl.DataFrame(test_super.astype(np.float64), schema=predict_cols)
)

out_dir = ROOT / "submissions"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "superblend.parquet"
sub.write_parquet(out_path)
print("Saved:", out_path)
print("shape:", sub.shape)


In [ ]:
while True:
    continue